<img src="images/28.png" width="40%">

<img src="images/29.png" width="40%">

<img src="images/31.png" width="40%">

<img src="images/32.png" width="40%">

<img src="images/33.png" width="40%">

<img src="images/34.png" width="40%">

<img src="images/35.png" width="40%">

<img src="images/36.png" width="40%">

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class MultiHeadCausalAttention(nn.Module):
    def __init__(self, n_embd, n_head, bias=False):
        super().__init__()
        self.n_embd = n_embd
        self.n_head = n_head
        assert n_embd % n_head == 0, "嵌入维度必须可以被头数整除"
        self.d_k = n_embd // n_head   # 每个头的维度 d_k

        # QKV合并投影，对应nanoGPT c_attn
        self.c_attn = nn.Linear(n_embd, 3 * n_embd, bias=bias)
        # 多头输出投影 W_O，对应 c_proj
        self.c_proj = nn.Linear(n_embd, n_embd, bias=bias)

        # 因果下三角mask缓冲区，不是可训练参数
        self.register_buffer(
            "bias",
            torch.tril(torch.ones(1, 1, 1024, 1024))
        )

    def forward(self, x):
        B, T, C = x.shape   # B batch, T序列长度, C = n_embd
        # 1. QKV一次性投影
        qkv = self.c_attn(x)          # [B, T, 3*C]
        q, k, v = qkv.split(self.n_embd, dim=-1) # 切分得到Q K V  [B,T,C]

        # 2. 分头：拆成多头 B,n_head,T,d_k
        q = q.view(B, T, self.n_head, self.d_k).transpose(1, 2) # [B, nh, T, dk]
        k = k.view(B, T, self.n_head, self.d_k).transpose(1, 2)
        v = v.view(B, T, self.n_head, self.d_k).transpose(1, 2)

        # 3. 缩放点积注意力 S = QK^T / sqrt(d_k)
        att_score = q @ k.transpose(-2, -1) * (1.0 / torch.sqrt(torch.tensor(self.d_k, dtype=torch.float32)))
        # 4. 因果mask：把上三角(未来token)填充 -inf
        att_score = att_score.masked_fill(self.bias[:,:,:T,:T]==0, float("-inf"))
        # 5. softmax得到注意力权重A
        att_weight = F.softmax(att_score, dim=-1)
        # 6. A @ V 加权聚合value
        y = att_weight @ v   # [B, nh, T, dk]

        # 7. concat多头：把多个头拼回原始嵌入维度
        y = y.transpose(1,2).contiguous().view(B, T, C) # [B, T, C]
        # 8. 输出投影 W_O
        y = self.c_proj(y)
        return y


# ------------------- 测试运行 -------------------
if __name__ == "__main__":
    B = 2      # batch size
    T = 8      # 序列长度 block_size=8
    n_embd = 128
    n_head = 4

    mha = MultiHeadCausalAttention(n_embd=n_embd, n_head=n_head)
    x = torch.randn(B, T, n_embd)   # 模拟输入token特征向量 [B,T,n_embd]

    out = mha(x)
    print(f"输入x shape: {x.shape}")
    print(f"多头注意力输出 shape: {out.shape}")

    # 打印中间张量shape观察
    print("\n关键shape流转：")
    print(f"输入 x           → [B, T, n_embd]      {x.shape}")
    print(f"分头之后 q/k/v   → [B, nh, T, d_k]    {torch.randn(B,n_head,T,n_embd//n_head).shape}")
    print(f"att_score        → [B, nh, T, T]      {torch.randn(B,n_head,T,T).shape}")
    print(f"concat后 y       → [B, T, n_embd]     {out.shape}")


输入x shape: torch.Size([2, 8, 128])
多头注意力输出 shape: torch.Size([2, 8, 128])

关键shape流转：
输入 x           → [B, T, n_embd]      torch.Size([2, 8, 128])
分头之后 q/k/v   → [B, nh, T, d_k]    torch.Size([2, 4, 8, 32])
att_score        → [B, nh, T, T]      torch.Size([2, 4, 8, 8])
concat后 y       → [B, T, n_embd]     torch.Size([2, 8, 128])
